# Anapneo — sleep-apnea dashboard on Google Colab

Audio-only sleep-apnea screening: a **CNN + CRNN ensemble** (trained on the
APSAA full-night dataset) wrapped in the **Flask dashboard** from the repo.

This notebook runs the *whole* thing on a Colab GPU: the website, the uploaded
audio pipeline, the PDF report and the chat. The model weights
(`checkpoints/best_cnn.pt`, `best_crnn.pt`, `norm.json`) ship inside
`anapneo_colab.zip` — nothing is downloaded except the Python packages.

**Steps:** run cells 1 → 7 in order. Cell 3 downloads the bundle
(`anapneo_colab.zip`, ~13 MB) automatically from GitHub
(`ayaanmaheshwari/anapneo-colab`) — no upload needed. Cell 7 keeps running
while you use the dashboard; stop it to shut the server.

For the AI-written report and chat, paste a free Google **Gemini API key** into
the key field in the dashboard header (it stays in your browser). No local LLM
is needed; with no key the deterministic built-in report/answers are used.


In [ ]:
#@title 1 · Check the GPU
import sys
import torch

print(sys.version.split()[0])
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("No GPU — CPU works too, just slower. Runtime > Change runtime type > T4 GPU.")


In [ ]:
#@title 2 · Install dependencies
# Colab already ships torch, pandas, numpy, matplotlib, scikit-learn,
# soundfile, librosa and Flask; this is just a safety net and stays fast.
%pip install -q flask soundfile librosa pyarrow


In [ ]:
#@title 3 · Download the project bundle (automatic)
import os
import urllib.request
import zipfile
from pathlib import Path

# The bundle (code + trained checkpoints + demo clips, ~13 MB) lives here:
ZIP_URL = "https://raw.githubusercontent.com/ayaanmaheshwari/anapneo-colab/main/anapneo_colab.zip"
ZIP_PATH = Path("/content/anapneo_colab.zip")
PROJECT = Path("/content/anapneo")
MIN_ZIP_BYTES = 1_000_000


def _good(p):
    try:
        return (p.is_file() and p.stat().st_size >= MIN_ZIP_BYTES
                and zipfile.is_zipfile(p))
    except OSError:
        return False


if not (PROJECT / "colab_serve.py").exists():
    # 1) a valid local copy already? (Drive or a previous upload)
    if not _good(ZIP_PATH):
        for p in (Path("/content/drive/MyDrive/anapneo_colab.zip"),
                  *Path("/content").glob("*.zip")):
            if _good(p):
                ZIP_PATH = p
                break

    # 2) otherwise download it straight from GitHub
    if not _good(ZIP_PATH):
        print("Downloading anapneo_colab.zip from GitHub ...")
        try:
            urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
            print(f"  got {ZIP_PATH.stat().st_size:,} bytes")
        except Exception as exc:
            print("  download failed:", exc)

    # 3) last resort: manual upload
    if not _good(ZIP_PATH):
        from google.colab import files
        print("Upload anapneo_colab.zip manually:")
        files.upload()
        ZIP_PATH = Path("/content/anapneo_colab.zip")
        if not _good(ZIP_PATH):
            raise SystemExit("No usable bundle — check your connection and retry.")

    print(f"Extracting {ZIP_PATH.name} ({ZIP_PATH.stat().st_size:,} bytes) ...")
    with zipfile.ZipFile(ZIP_PATH) as z:
        bad = z.testzip()
        if bad:
            raise SystemExit(f"Corrupt member {bad} — delete the zip and re-run.")
        z.extractall("/content")

os.chdir(PROJECT)
print("Project:", Path.cwd())
print(sorted(p.name for p in Path.cwd().iterdir() if p.is_dir()))


In [ ]:
#@title 4 · Verify the bundled models
from pathlib import Path

from src import config

ckpts = sorted(p.name for p in config.CHECKPOINT_DIR.glob("*.pt"))
print("project root :", config.PROJECT_ROOT)
print("models (ckpt):", ckpts)
print("norm saved   :", (config.CHECKPOINT_DIR / "norm.json").is_file())
print("demo clips   :", [p.name for p in sorted(Path("showcase/clips").glob("*.wav"))])
assert ckpts, "No model checkpoints — re-run cell 3 with the anapneo_colab.zip bundle."
print("\nReady.")


In [ ]:
#@title 5 · Smoke test — analyse a 20-minute clip (~15 s)
from pathlib import Path

from webapp.analysis import analyze

clip = sorted(Path("showcase/clips").glob("apnea*.wav"))[0]
result = analyze(clip, progress=lambda frac, msg: None)
print("device     :", result["model"]["device"])
print("threshold  :", result["model"]["threshold"])
print("KPI        :", result["kpi"])


In [ ]:
#@title 6 · Optional — full-night recordings from Google Drive
# Optional: use the real 8-hour APSAA nights.
# Put the APSAA subject folders in Drive (one folder per subject, containing
# <SUBJECT>.wav) and point SRC at them. They are symlinked into data/, so the
# dashboard's pre-loaded samples light up — nothing is copied.
from pathlib import Path

from google.colab import drive

SRC = Path("/content/drive/MyDrive/APSAA")      # <-- change if yours differs

drive.mount("/content/drive")
if not SRC.is_dir():
    print("Not found:", SRC, "— skipping (the demo clips still work).")
else:
    data_dir = Path("data")
    data_dir.mkdir(exist_ok=True)
    added = 0
    for sub in sorted(SRC.iterdir()):
        if sub.is_dir() and not (data_dir / sub.name).exists():
            (data_dir / sub.name).symlink_to(sub, target_is_directory=True)
            added += 1
    print(f"Linked {added} subject folder(s) into data/ — refresh the dashboard.")


In [ ]:
#@title 7 · Launch the dashboard (keep this cell running)
import importlib
import os

# No local LLM: chat uses your Gemini key from the UI, else the built-in
# answers. Keep this "off" so nothing tries to download a 4.7 GB model.
os.environ["ANAPNEO_LLM"] = "off"

import colab_serve
importlib.reload(colab_serve)

# offline=False allows the Gemini report/chat when a key is pasted in the header
# (Colab has internet). Set offline=True for a fully sealed, key-free run.
colab_serve.serve(port=5000, offline=False, embed=True)


## Using it

- **Samples** → "Quick demo — Apnea clip" or "Normal clip" gives a full result
  in ~15 s on a GPU. If you linked Drive in cell 6, the three full-night samples
  appear too (a few minutes each).
- **Upload** your own `.wav` / `.mp3` with the drop zone. Through the Colab
  proxy, very large uploads can time out — prefer < ~200 MB, or copy files to
  `/content` and drop them into the uploads folder.
- **Gemini key** → paste it in the "API key" field in the dashboard header.
  The summary and the follow-up chat then use Gemini; the PDF always works.
  Without a key everything falls back to the deterministic built-in text, and
  the source pill shows `Offline · built-in`.
- **Stop** the server by interrupting cell 7 (■). Colab's proxy URL stops
  working when the cell stops.

### If cell 3 cannot download the bundle

Cell 3 fetches `anapneo_colab.zip` from
`https://raw.githubusercontent.com/ayaanmaheshwari/anapneo-colab/main/anapneo_colab.zip`.
If that fails, it will ask you to upload the file manually — drag it into the
**Files** pane (left sidebar) and re-run the cell. To force a clean retry:

```python
!rm -rf /content/anapneo /content/anapneo_colab.zip
```

### Sharing / collaborating

Share the notebook with *File → Share* (viewers can run their own copy), or
`File → Save a copy in Drive` and share that. Each person runs the same cells
on their own Colab runtime — the model weights come from the bundle, and each
person can paste their own Gemini key.

> Research prototype only: audio cannot measure airflow or SpO₂, so this is a
> pre-screening aid and never a diagnosis.
